# Lab 22-12 · Power Flow and Frequency Response

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import grid


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Power flow in volts and ohms

*Predict* P, Q1 and Q2 for Example 22-12.4.

In [2]:
p, q1, q2 = grid.two_bus(1.0, 0.98, 0.1, 10)
print(f"per unit: P = {p:.3f} ({p*100:.0f} MW), Q1 = {q1:.3f}, Q2 = {q2:.3f}")
s = ngspice("powerflow.cir")
print({kk: s[kk] for kk in ("pmw", "q1", "q2")})

per unit: P = 1.702 (170 MW), Q1 = 0.349, Q2 = 0.047
{'pmw': 170.176, 'q1': 0.3488855, 'q2': 0.04711617}


## Part 2 · Frequency response

*Predict* the initial RoCoF and the settled deviation; *guess* the nadir.

In [3]:
t, df = grid.frequency_response(1800, 4, 50e3)
print(f"RoCoF {grid.rocof(1800, 4, 50e3):.2f} Hz/s; nadir {-df.min():.3f} Hz at {t[df.argmin()]:.2f} s; settles at {df[-1]:.3f} Hz")
s = ngspice("freq.cir")
print({kk: s[kk] for kk in ("rocof0", "nadir", "tnadir", "fss")})

RoCoF 0.27 Hz/s; nadir 0.344 Hz at 2.27 s; settles at -0.102 Hz
{'rocof0': 0.2695885, 'nadir': 0.3438955, 'tnadir': 2.27329, 'fss': -0.101877}


## Part 3 · Less inertia

*Predict* the nadir and its time with H = 2 s.

In [4]:
plt.figure()
for h in (4, 2):
    t, df = grid.frequency_response(1800, h, 50e3, t_end=30, n=3001)
    plt.plot(t, 60 + df, label=f"H = {h} s")
    print(f"H = {h} s: nadir {-df.min():.3f} Hz at {t[df.argmin()]:.2f} s")
plt.xlabel("s"); plt.ylabel("Hz"); plt.legend(); plt.show()

H = 4 s: nadir 0.344 Hz at 2.27 s
H = 2 s: nadir 0.450 Hz at 1.52 s


## Part 4 · Sweeps

*Predict* where P peaks, and when the ground fault exceeds the three-phase fault.

In [5]:
d = np.linspace(0, 120, 241)
plt.figure(); plt.plot(d, [grid.two_bus(1.0, 0.98, 0.1, x)[0] for x in d]); plt.xlabel("delta (deg)"); plt.ylabel("P (pu)"); plt.show()
ratios = np.linspace(0.25, 3, 12)
for r in ratios[::3]:
    print(f"Z0/Z1 = {r:.2f}: SLG/3ph = {grid.fault_slg(0.3, 0.3, 0.3*r)/grid.fault_3ph(0.3):.2f}")

Z0/Z1 = 0.25: SLG/3ph = 1.33
Z0/Z1 = 1.00: SLG/3ph = 1.00
Z0/Z1 = 1.75: SLG/3ph = 0.80
Z0/Z1 = 2.50: SLG/3ph = 0.67


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.